# Fusion

## 1. Imports et premières informations

In [50]:
import numpy as np
import pandas as pd
import os

In [51]:
df_allocine = pd.read_csv("../0_Data/1_Raw/data_allocine.csv")
df_imdb = pd.read_csv("../0_Data/1_Raw/data_imdb.csv")

display(df_allocine.head())
print()
display(df_imdb.head())

,notesAllocine,avis,dates,series,allocine_id,review_id,plateforme
0,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms,28230,282301,Allocine
1,5.0,Très bonne série. Des épisodes trop court tr...,2026-02-07,A Knight of the Seven Kingdoms,28230,282302,Allocine
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms,28230,282303,Allocine
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,A Knight of the Seven Kingdoms,28230,282304,Allocine
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,A Knight of the Seven Kingdoms,28230,282305,Allocine


,notesImdb,avis,dates,series
0,8,Honest Opinion - Turns out Westeros is way mor...,2026-01-24,A Knight of the Seven Kingdoms
1,9,"So Far, So Good (Don't Screw It Up) - The firs...",2026-01-19,A Knight of the Seven Kingdoms
2,9,A proper return to Westeros - I stayed up for ...,2026-01-19,A Knight of the Seven Kingdoms
3,9,The only disappointment i had with ep 1 - Its ...,2026-01-19,A Knight of the Seven Kingdoms
4,9,Really freaking good with only one bad moment ...,2026-01-19,A Knight of the Seven Kingdoms


In [52]:
print(df_allocine.info())
print()
print(df_imdb.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   notesAllocine  210 non-null    float64
 1   avis           210 non-null    object 
 2   dates          210 non-null    object 
 3   series         210 non-null    object 
 4   allocine_id    210 non-null    int64  
 5   review_id      210 non-null    int64  
 6   plateforme     210 non-null    object 
dtypes: float64(1), int64(2), object(4)
memory usage: 11.6+ KB
None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1412 entries, 0 to 1411
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   notesImdb  1412 non-null   int64 
 1   avis       1412 non-null   object
 2   dates      1412 non-null   object
 3   series     1412 non-null   object
dtypes: int64(1), object(3)
memory usage: 44.3+ KB
None


## 2. Concaténation et transformations

In [53]:
df_concat = pd.concat([df_allocine, df_imdb], axis=0).reset_index(drop=True)
df_concat

,notesAllocine,avis,dates,series,allocine_id,review_id,plateforme,notesImdb
0,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms,28230.0,282301.0,Allocine,NaN
1,5.0,Très bonne série. Des épisodes trop court tr...,2026-02-07,A Knight of the Seven Kingdoms,28230.0,282302.0,Allocine,NaN
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms,28230.0,282303.0,Allocine,NaN
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,A Knight of the Seven Kingdoms,28230.0,282304.0,Allocine,NaN
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,A Knight of the Seven Kingdoms,28230.0,282305.0,Allocine,NaN
...,...,...,...,...,...,...,...,...
1617,NaN,Mediocre filmmaking - Weak script in combinati...,2025-10-29,King & Conqueror,NaN,NaN,NaN,1.0
1618,NaN,drunk history - As many other reviewers have n...,2026-02-02,King & Conqueror,NaN,NaN,NaN,3.0
1619,NaN,I get it... -,2026-02-26,King & Conqueror,NaN,NaN,NaN,7.0
1620,NaN,Realism... - Earl's military crews count 5-8 s...,2026-03-02,King & Conqueror,NaN,NaN,NaN,3.0


In [54]:
# Création de la colonne note unique sur une même échelle
df_concat['notes'] = (df_concat['notesAllocine'].fillna(0) * 2).astype(int) \
    + df_concat['notesImdb'].fillna(0)

# Conservation du nom de la plateforme
df_concat['plateforme'] = np.where(df_concat["notesAllocine"].isna(), 'Imdb', 'Allocine')

# Dates en format 'date'
df_concat['dates'] = pd.to_datetime(df_concat['dates'])

# Supprimer les colonnes 'notesAllocine' et 'notesImdb' plus utiles
df_concat = df_concat.drop(['notesAllocine', 'notesImdb'], axis=1)

In [55]:
# remplir les NaN de la colonne 'allocine_id'
df_concat['allocine_id'] = df_concat.groupby('series')['allocine_id'].ffill()

print(df_concat['allocine_id'].value_counts(), '\n') 

allocine_id
28230.0    1404
35604.0     218
Name: count, dtype: int64 



In [56]:
# Création du review_id, qui correspond au code séries suivi de l'index
for serie in df_concat['series'].unique():
    for idx, value in enumerate(df_concat['review_id']):
        if pd.isna(value):
            # iloc cherche la position et non l'index (parfois, l'index ne reflète pas la position)
            pos = df_concat.index.get_loc(idx)
            # review_id de la ligne précédent + 1
            df_concat.loc[idx, 'review_id'] = df_concat['review_id'].iloc[pos-1] + 1

# Retrier par index
df_concat = df_concat.sort_index()

print(df_concat['review_id'].value_counts())

# Pas de NaN ni de valeurs duppliquées 

review_id
282301.0      1
282302.0      1
282303.0      1
282304.0      1
282305.0      1
             ..
35605618.0    1
35605619.0    1
35605620.0    1
35605621.0    1
35605622.0    1
Name: count, Length: 1622, dtype: int64


In [57]:
# review_id, allocine_id et notes en int
df_concat[['review_id', 'allocine_id', 'notes']] = df_concat[['review_id', 'allocine_id', 'notes']].astype(int)

In [58]:
df_concat[df_concat['series'] == 'A Knight of the Seven Kingdoms']

,avis,dates,series,allocine_id,review_id,plateforme,notes
0,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms,28230,282301,Allocine,9
1,Très bonne série. Des épisodes trop court tr...,2026-02-07,A Knight of the Seven Kingdoms,28230,282302,Allocine,10
2,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms,28230,282303,Allocine,10
3,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,A Knight of the Seven Kingdoms,28230,282304,Allocine,10
4,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,A Knight of the Seven Kingdoms,28230,282305,Allocine,10
...,...,...,...,...,...,...,...
1444,"The comeback of GOT - Juste imagine, you have ...",2026-02-23,A Knight of the Seven Kingdoms,28230,35605445,Imdb,10
1445,A Beauty of Westeros - I have a close friend w...,2026-02-23,A Knight of the Seven Kingdoms,28230,35605446,Imdb,10
1446,Old stories - I feel like this is not a series...,2026-02-23,A Knight of the Seven Kingdoms,28230,35605447,Imdb,4
1447,Very bad show - I don't understand the message...,2026-02-20,A Knight of the Seven Kingdoms,28230,35605448,Imdb,1


In [59]:
# Export en csv
display(df_concat.head())

# Un point = dossier actuel, deux points = dossier parent
output_path = '../0_Data/2_Clean/Traitement_intermediaire/data_clean.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_concat.to_csv(output_path, index=False)


,avis,dates,series,allocine_id,review_id,plateforme,notes
0,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms,28230,282301,Allocine,9
1,Très bonne série. Des épisodes trop court tr...,2026-02-07,A Knight of the Seven Kingdoms,28230,282302,Allocine,10
2,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms,28230,282303,Allocine,10
3,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,A Knight of the Seven Kingdoms,28230,282304,Allocine,10
4,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,A Knight of the Seven Kingdoms,28230,282305,Allocine,10
